In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report



In [2]:

# CSV mit Semikolon laden (deutsches Format)
data = pd.read_csv("Ein_Ausgaben_privat.csv", sep=";")

# Beträge bereinigen: Leerzeichen, €, Punkt (Tausendertrennzeichen), Komma (Dezimal)
def parse_amount(s):
    if pd.isna(s):
        return None
    s = str(s).strip().replace("€", "").replace(".", "").replace(",", ".").strip()
    try:
        return float(s)
    except ValueError:
        print("Error")
        return None

data["amount_parsed"] = data["amount"].apply(parse_amount)
data.head()


,bookingDate,payee,purpose,amount,categoryMain,amount_parsed
0,30.12.25,Philipp Hardy,Kompensationszahlung gemäß Abschlussvereinbarung,"2.000,00 €",ES,2000.00
1,30.12.25,PayPal Europe S.a.r.l. et Cie S.C.A ...,"1047237348812/PP.6580.PP/. , Ihr Einkauf bei","-55,00 €",V,-55.00
2,30.12.25,Vodafone GmbH Ferdinand-Braun-Platz 1,Kd.Nr. 001963060871 VK 1057505887 Rg.Nr. 00377...,"-19,99 €",F,-19.99
3,29.12.25,StratoDE/Berlin,VISA Debitkartenumsatz,"-6,00 €",F,-6.00
4,29.12.25,StratoDE/Berlin,VISA Debitkartenumsatz,"-6,00 €",F,-6.00


In [3]:
data["amount_parsed"] = data["amount"].apply(parse_amount)
data["text"] = data["payee"].fillna("") + " " + data["purpose"].fillna("")
data = data.dropna(subset=["categoryMain", "amount_parsed"])
y = data["categoryMain"]


In [ ]:
X_text_raw_train, X_text_raw_test, y_train, y_test = train_test_split(
    data["text"], y, test_size=0.2, random_state=42
)
train_idx = X_text_raw_train.index
test_idx  = X_text_raw_test.index
amount_train = data.loc[train_idx, "amount_parsed"].values.reshape(-1, 1)
amount_test  = data.loc[test_idx, "amount_parsed"].values.reshape(-1, 1)


In [5]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=500)
X_text_train = vectorizer.fit_transform(X_text_raw_train)
X_text_test  = vectorizer.transform(X_text_raw_test)


In [6]:
scaler = StandardScaler()
amount_train_scaled = scaler.fit_transform(amount_train)
amount_test_scaled  = scaler.transform(amount_test)


In [7]:
X_train = hstack([X_text_train, csr_matrix(amount_train_scaled)], format="csr")
X_test  = hstack([X_text_test,  csr_matrix(amount_test_scaled)],  format="csr")


In [8]:
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

          AB       0.00      0.00      0.00         0
          AT       1.00      1.00      1.00         1
         AUS       1.00      1.00      1.00         7
           E       0.80      0.84      0.82        19
          ES       1.00      1.00      1.00         1
           F       0.83      0.83      0.83        18
           G       1.00      1.00      1.00         1
           M       1.00      0.33      0.50         3
          MW       0.33      0.50      0.40         2
           U       1.00      0.50      0.67         2
           V       0.61      0.61      0.61        23

    accuracy                           0.75        77
   macro avg       0.78      0.69      0.71        77
weighted avg       0.78      0.75      0.76        77



/Users/philipphardy/Documents/Github/venera/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/philipphardy/Documents/Github/venera/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/philipphardy/Documents/Github/venera/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 